# 이미지와 레이어

Docker Desktop의 실습 전용 이미지와 스택만 사용해 결과를 확인합니다.

## 실행 환경 준비

Docker 데몬의 버전을 확인하고 공통 명령 함수를 준비합니다.

### 이 준비 셀이 하는 일

이 준비 셀은 현재 폴더와 부모 폴더에서 `Dockerfile.baseline`이 있는 Ch32 실습 폴더를 찾습니다. `run(*args, check=True)`은 모든 Docker 명령을 그 폴더에서 실행하고 표준 출력·오류와 종료 코드를 `CompletedProcess`로 반환합니다. 실패 자체를 관찰하는 셀만 `check=False`를 전달하며, 마지막에는 Docker Engine이 응답하는지 확인합니다.

In [1]:
# Docker 명령을 일정한 폴더에서 실행하고 실패도 결과로 받습니다.
from pathlib import Path
import json, subprocess, time, urllib.request
# 챕터 폴더와 notebooks 폴더 어디에서 시작해도 제공 자료를 찾습니다.
candidates = [parent / "labs/part7/ch32-docker" for parent in (Path.cwd(), *Path.cwd().parents)]
candidates += [Path.cwd(), Path.cwd().parent]
LAB = next(path for path in candidates if (path / "Dockerfile.baseline").is_file())
def run(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(args, cwd=LAB, text=True, capture_output=True, check=check)
version = run("docker", "version", "--format", "{{.Server.Version}}")
print("Docker Engine", version.stdout.strip())
assert version.stdout.strip()

Docker Engine 29.4.1


## 이미지의 읽기 전용 레이어 관찰

최적화 이미지의 기록을 크기와 명령으로 읽습니다.

### 바로 다음 코드를 읽는 방법

Dockerfile의 명령은 모두 이미지 기록에 남지만 역할은 다릅니다. `RUN`, `COPY`, `ADD`처럼 파일을 바꾸는 명령은 파일 시스템 레이어를 만들고, `CMD`와 `HEALTHCHECK`는 실행 기본값과 상태 검사 같은 메타데이터를 기록해 0B로 보일 수 있습니다. 이미지는 읽기 전용 레이어와 실행 설정을 담은 템플릿이고, 컨테이너는 그 위에 쓰기 가능한 층을 더한 실행 인스턴스입니다.

In [2]:
# 이미지가 없으면 공개 실습 Dockerfile로 먼저 빌드합니다.
run("docker", "build", "-f", "Dockerfile", "-t", "ch32-optimized:local", ".")
history = run("docker", "history", "--no-trunc", "--format", "{{.Size}}|{{.CreatedBy}}", "ch32-optimized:local")
rows = [line for line in history.stdout.splitlines() if line]
print("layer rows", len(rows))
print("\n".join(rows[:5]))
assert len(rows) >= 5

layer rows 19
0B|CMD ["uvicorn" "app:app" "--host" "0.0.0.0" "--port" "8000"]
0B|HEALTHCHECK {Test:[CMD-SHELL python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"] Interval:5s Timeout:2s StartPeriod:0s StartInterval:0s Retries:5}
0B|EXPOSE [8000/tcp]
0B|USER course
0B|ENV PATH=/app/.venv/bin:/usr/local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1


## 컨테이너의 쓰기 가능 층 확인

같은 이미지에서 만든 두 컨테이너의 변경은 서로 공유되지 않습니다.

In [3]:
# 임시 컨테이너에서 파일을 만들고 변경 내역을 확인합니다.
created = run("docker", "create", "ch32-optimized:local", "python", "-c", "open('/tmp/lesson.txt','w').write('layer')")
cid = created.stdout.strip()
run("docker", "start", "-a", cid)
diff = run("docker", "diff", cid)
print(diff.stdout.strip())
run("docker", "rm", cid)
assert "/tmp/lesson.txt" in diff.stdout

C /tmp
A /tmp/lesson.txt


## 실험 결과 해석

이미지 기록과 컨테이너 변경 기록은 서로 다른 범위를 보여 줍니다.

### 이 단계에서 구분할 범위

컨테이너 안에서 파일을 바꿔도 원본 이미지는 변하지 않습니다. 반대로 Dockerfile을 고쳐 다시 빌드하면 새 이미지 식별자가 생깁니다. 영속 데이터는 컨테이너의 쓰기 층에만 두지 않고 볼륨이나 외부 저장소로 분리해야 컨테이너 교체 뒤에도 남습니다.

In [4]:
# 이미지 자체에는 임시 파일이 추가되지 않았는지 새 컨테이너에서 확인합니다.
probe = run("docker", "run", "--rm", "ch32-optimized:local", "python", "-c", "import os; print(os.path.exists('/tmp/lesson.txt'))")
print("new container has file:", probe.stdout.strip())
assert probe.stdout.strip() == "False"

new container has file: False
